In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

In [ ]:
# ============================================================
# Imports
# ============================================================

from ablations import (
    ablation_1,
    ablation_2,
    ablation_3,
    ablation_4,
    ablation_5,
    ablation_6,
    build_ablation_suite,
)

# Optional: inspect generated configs before running training.
from ablations.model_factory import build_model_from_ablation_config


In [ ]:
# ============================================================
# Shared dataset config
# ============================================================
# For first real experiments, synthetic_long_context is the best controlled
# dataset because it stresses retrieval and long-context behavior.
#
# For natural language experiments, set:
#   dataset = "wikitext2"
#   dataset = "tinystories"
#   dataset = "ag_news"
#   dataset = "imdb"
#   dataset = "minipile"
#
# HF datasets may download/tokenize data. Synthetic does not.

DATA_CONFIG = {
    "dataset": "synthetic_long_context",

    # Sequence/data scale
    "block_size": 128,
    "batch_size": 4,
    "num_train_examples": 2_000,
    "num_val_examples": 300,

    # Synthetic retrieval structure
    "min_filler_tokens": 16,
    "max_filler_tokens": 96,
    "num_keys_per_example": 4,

    # Synthetic vocabulary controls.
    # Keep these modest for small models.
    "num_key_types": 64,
    "num_value_types": 64,
    "vocab_filler_size": 68,

    # Loader behavior
    "num_workers": 0,
}


In [ ]:
# ============================================================
# Model memory/VRAM limits
# ============================================================
# These are not the full architecture definition.
# They are upper-level knobs to keep variants inside your memory budget.
# The ablation itself still controls the feature being tested.

MAX_MODEL = {
    "d_model": 128,
    "n_layers": 4,
    "max_seq_len": 128,

    "n_heads": 4,
    "head_dim": 32,
    "rotary_dim": 32,

    "mlp_hidden_dim": 512,

    # MoE variants use these when enabled.
    "num_experts": 4,
    "top_k_experts": 2,
    "expert_hidden_dim": 256,
    "shared_hidden_dim": 256,

    # mHC variants use this when enabled.
    "n_hc": 4,

    # MTP variants use this when enabled.
    "mtp_hidden_dim": 128,
}


In [ ]:
# ============================================================
# Training budget
# ============================================================
# For a real GPU run, device="cuda".
# For CPU smoke, use device="cpu", max_batches_per_epoch=1 or 2.

TRAINING_CONFIG = {
    "epochs": 1,
    "max_batches_per_epoch": 30,
    "eval_max_batches": 10,

    # "adamw" is lighter and good for quick ablations.
    # "muon_adamw" is closer to the project training stack.
    "optimizer_type": "muon_adamw",

    "learning_rate": 3e-4,
    "min_learning_rate": 3e-5,
    "weight_decay": 0.1,
    "grad_clip_norm": 1.0,
    "grad_accum_steps": 1,
    "warmup_steps": 1,

    "device": "cuda",      # change to "cpu" for smoke tests
    "amp_enabled": True,   # set False on CPU
    "amp_dtype": "bf16",

    # Keep logs quiet in notebook unless debugging.
    "log_every": 0,
    "module_metrics_every": None,

    # Important: runner saves before clearing model/cache.
    "save_checkpoints": True,

    # Runs generation/cache benchmark after eval.
    "run_inference_benchmark": True,
    "inference_max_new_tokens": 8,
}


In [ ]:
# ============================================================
# Dry-run: inspect configs without training
# ============================================================

runs = build_ablation_suite(
    "A1",
    data_config=DATA_CONFIG,
    max_model=MAX_MODEL,
    training_config=TRAINING_CONFIG,
    seeds=[1],
    quick=False,
)

print("num_runs:", len(runs))
print("first_variant:", runs[0]["variant_name"])
print("output_dir:", runs[0]["output_dir"])
print("model_config:", runs[0]["model_config"])


In [ ]:
# ============================================================
# Optional: instantiate one model only to inspect size
# ============================================================

model = build_model_from_ablation_config(runs[0])
num_params = sum(p.numel() for p in model.parameters())

print("variant:", runs[0]["variant_name"])
print("num_params:", f"{num_params:,}")

del model


In [ ]:
# ============================================================
# Run Ablation 1
# ============================================================
# A1 answers:
# MHA vs HCA vs CSA vs hybrid attention.

results = ablation_1(
    data_config=DATA_CONFIG,
    max_model=MAX_MODEL,
    training_config=TRAINING_CONFIG,
    seeds=[1],
    quick=False,
    base_output_dir="outputs/ablations",
)

len(results), results[0]["output_dir"]


In [ ]:
# ============================================================
# Other suites
# ============================================================

# A2: compression/window grid
# results = ablation_2(DATA_CONFIG, MAX_MODEL, TRAINING_CONFIG, seeds=[1])

# A3: mHC utility
# results = ablation_3(DATA_CONFIG, MAX_MODEL, TRAINING_CONFIG, seeds=[1])

# A4: MoE routing/shared experts/balance loss
# results = ablation_4(DATA_CONFIG, MAX_MODEL, TRAINING_CONFIG, seeds=[1])

# A5: MTP depth/weight
# results = ablation_5(DATA_CONFIG, MAX_MODEL, TRAINING_CONFIG, seeds=[1])

# A6: full stack additive composition
# results = ablation_6(DATA_CONFIG, MAX_MODEL, TRAINING_CONFIG, seeds=[1])
